# SupeRR Selector Season 2 — End-to-End Audited Pressure-Performance Pipeline (v5 Final)
**100% Executable on Official Competition CSVs | Pure Shrunk Residual $R_{i,\text{EB}}$ + Standalone Reliability Metrics ($C_i, D_i$)**

### Methodological Architecture:
1. **Data Hygiene & Competition Scope**: Excludes the 6 Super Over tie-breakers (38 balls, audited separately) and minor/Associate fixtures, retaining IPL and Full-Member T20Is with exact legal-ball counting and non-linear DLS resource scaling $(\text{inn\_total\_balls}/120)^{0.85}$.
2. **Fold-Local Pure Match-State Win Probability Model**: Trains separate Innings-1 and Innings-2 models using 5-fold match-grouped cross-validation (`GroupKFold`), fitting the par table, terminal Innings-1 logistic boundary, and Dirichlet-smoothed wickets-aware outcome distributions $P(o \mid \text{Format}, \text{Phase}, \text{WicketsBin}, \text{RunsNeededBin})$ strictly inside each training fold.
3. **Symmetric Batting & Bowling Attribution**: Credits batters for bat runs scored off No-Balls (`Overstep`, with `bl_drop = 0`) and evaluates non-bowler / non-striker run-outs via `wp_after_nowkt` (`w = 0`) so run contributions are preserved without crediting run-out wickets to bowlers.
4. **Empirical Bayes Baseline-Adjusted Residual ($R_{i,\text{EB}}$) & Standalone Consistency ($C_i, D_i$)**: Ranks players by their posterior baseline-adjusted residual $R_{i,\text{EB}}$ (avoiding arbitrary composite weights) while reporting exposure-adjusted positive-match frequency $C_{i,\text{EB}}$ (%) and Sortino downside semi-deviation $D_{i,\text{EB}}$ alongside `reliability_score`.

In [1]:
import gc, warnings, json
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.special import expit
from scipy.optimize import minimize
from scipy.stats import spearmanr

warnings.filterwarnings("ignore")

DATA = Path(r"C:\Users\Anmol Sriwastava\Desktop\RR_Analytics\data") if Path(r"C:\Users\Anmol Sriwastava\Desktop\RR_Analytics\data").exists() else (Path("/working_dir/c_e74e0d7d2df6d8f4/user_uploaded_files/ttl=63d/user_uploaded") if Path("/working_dir/c_e74e0d7d2df6d8f4/user_uploaded_files/ttl=63d/user_uploaded").exists() else Path("./data"))
OUT  = Path(r"C:\Users\Anmol Sriwastava\Desktop\RR_Analytics\Analysis_v4_final") if Path(r"C:\Users\Anmol Sriwastava\Desktop\RR_Analytics").exists() else Path("./Analysis_v4_final")
OUT.mkdir(exist_ok=True, parents=True)
OUT.mkdir(exist_ok=True, parents=True)

# ══════════════════════════════════════════════════════════════════════════
# CELL 1: STEP 1, 2 & 3 — MATCH STATE, NON-LINEAR DLS, TERMINAL INN-1,
#         WICKETS-AWARE HIERARCHICAL OUTCOMES & PURE-STATE WP MODEL
# ══════════════════════════════════════════════════════════════════════════
def group_kfold_split(groups, n_splits=5):
    unique_groups, group_counts = np.unique(groups, return_counts=True)
    order = np.argsort(-group_counts, kind="mergesort")
    unique_groups = unique_groups[order]
    group_counts = group_counts[order]
    n_samples_per_fold = np.zeros(n_splits)
    group_to_fold = {}
    for g, c in zip(unique_groups, group_counts):
        lightest = np.argmin(n_samples_per_fold)
        n_samples_per_fold[lightest] += c
        group_to_fold[g] = lightest
    fold_indices = pd.Series(groups).map(group_to_fold).values
    for f in range(n_splits):
        te = np.where(fold_indices == f)[0]
        tr = np.where(fold_indices != f)[0]
        yield tr, te

def calc_auc(y_true, y_prob):
    y_true = np.asarray(y_true, dtype=int)
    y_prob = np.asarray(y_prob, dtype=float)
    order = np.argsort(y_prob)
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(y_prob) + 1)
    _, inv, counts = np.unique(y_prob, return_inverse=True, return_counts=True)
    sum_ranks = np.bincount(inv, weights=ranks)
    avg_ranks = sum_ranks / counts
    ranks = avg_ranks[inv]
    n1 = np.sum(y_true == 1)
    n0 = len(y_true) - n1
    return (np.sum(ranks[y_true == 1]) - n1 * (n1 + 1) / 2.0) / (n1 * n0)

def calc_logloss(y_true, y_prob, eps=1e-7):
    p = np.clip(y_prob, eps, 1 - eps)
    return -np.mean(y_true * np.log(p) + (1 - y_true) * np.log(1 - p))

def calc_brier(y_true, y_prob):
    return float(np.mean((y_true - y_prob) ** 2))

# ── STEP 1: LOAD & CLEAN (Separate Super Overs into dedicated audit table per Problem 5) ──
keep_cols = [
    "match_id", "innings", "overid", "innings_ball_num",
    "batting_team_id", "bowling_team_id",
    "striker_id", "striker_name", "bowler_id", "bowler_name",
    "runs", "is_wicket", "dismissal_type", "dismissed_player_id",
    "extras_type", "extras_2_runs", "pace_or_spin"
]
bbb_all = pd.read_csv(DATA / "ball_by_ball.csv", usecols=keep_cols)
super_overs = bbb_all[bbb_all["innings"].isin([3, 4])].copy()
bbb = bbb_all[bbb_all["innings"].isin([1, 2])].copy()
del bbb_all; gc.collect()

matches = pd.read_csv(DATA / "matches.csv", usecols=[
    "match_id", "comp_id", "match_date", "stage", "knock_out",
    "revised_target", "first_inn_revised_overs", "second_inn_revised_overs",
    "venue_id", "win_team_id"
])
comps = pd.read_csv(DATA / "competitions.csv", usecols=["comp_id", "comp_name", "match_format", "season"])

df = bbb.merge(matches, on="match_id", how="left").merge(comps, on="comp_id", how="left")
del bbb; gc.collect()
df = df.sort_values(["match_id", "innings", "innings_ball_num"]).reset_index(drop=True)

df["ext_str"] = df["extras_type"].fillna("none")
df["legal_ball"] = ~df["ext_str"].isin(["wide", "Overstep"])
df["is_wide"] = df["ext_str"] == "wide"
df["is_noball"] = df["ext_str"] == "Overstep"
df["is_bye_legbye"] = df["ext_str"].isin(["byes", "legbyes"])

df["extra_run"] = df["ext_str"].isin(["wide", "Overstep"]).astype(int)
df["approx_runs"] = (
    df["runs"].fillna(0).astype(int)
    + df["extra_run"]
    + df["extras_2_runs"].fillna(0).astype(int)
    + df["is_bye_legbye"].astype(int)
)
df["runs_bat"] = df["runs"].fillna(0).astype(int)
df["striker_out"] = (df["is_wicket"].astype(bool)) & (df["dismissed_player_id"] == df["striker_id"])

df["bowler_runs"] = df["runs"].fillna(0).astype(int) + df["extra_run"] + df["extras_2_runs"].fillna(0).astype(int)
df.loc[df["is_bye_legbye"], "bowler_runs"] = 0

non_bowler_dismissals = ["RunOut", "Retire Out", "Obstructing the field", "Obstracting"]
df["bowler_wkt"] = (df["is_wicket"].astype(bool)) & (~df["dismissal_type"].isin(non_bowler_dismissals))
df["is_t20i"] = (df["match_format"] == "T20I").astype(int)

# ── STEP 2: RECONSTRUCT MATCH STATE (NON-LINEAR DLS RESOURCE ADJUSTMENT) ──
df["inn_total_balls"] = 120
r1 = df["first_inn_revised_overs"]; r2 = df["second_inn_revised_overs"]
ok1 = (df["innings"] == 1) & r1.between(1, 120)
ok2 = (df["innings"] == 2) & r2.between(1, 120)
df.loc[ok1, "inn_total_balls"] = r1[ok1].round().astype(int)
df.loc[ok2, "inn_total_balls"] = r2[ok2].round().astype(int)

g = df.groupby(["match_id", "innings"], sort=False)
df["score_before"] = g["approx_runs"].cumsum() - df["approx_runs"]
df["wkts_before"] = (g["is_wicket"].cumsum().astype(int) - df["is_wicket"].astype(int)).clip(0, 9)
df["legal_before"] = g["legal_ball"].cumsum().astype(int) - df["legal_ball"].astype(int)
del g; gc.collect()

df["balls_left_before"] = (df["inn_total_balls"] - df["legal_before"]).clip(lower=0)
df["score_after"] = df["score_before"] + df["approx_runs"]
df["wkts_after"]  = (df["wkts_before"] + df["is_wicket"].astype(int)).clip(upper=10)
df["legal_after"] = df["legal_before"] + df["legal_ball"].astype(int)
df["balls_left_after"] = np.where(df["wkts_after"] >= 10, 0, (df["inn_total_balls"] - df["legal_after"]).clip(lower=0))

first_total = df[df["innings"] == 1].groupby("match_id")["approx_runs"].sum().rename("first_total")
df = df.merge(first_total, on="match_id", how="left")

df["target"] = np.nan
df.loc[df["innings"] == 2, "target"] = df.loc[df["innings"] == 2, "first_total"] + 1
df.loc[(df["innings"] == 2) & df["revised_target"].notna(), "target"] = df.loc[(df["innings"] == 2) & df["revised_target"].notna(), "revised_target"]

df["runs_needed_before"] = df["target"] - df["score_before"]
df["runs_needed_after"]  = df["target"] - df["score_after"]
df["bat_team_won"] = (df["win_team_id"] == df["batting_team_id"]).astype(float)
df.loc[df["win_team_id"].isna(), "bat_team_won"] = np.nan
df["phase"] = pd.cut(df["overid"].clip(1, 20), [0, 6, 15, 20], labels=[0, 1, 2]).astype(int)

# ── STEP 3: FOLD-LOCAL PAR GRID, NON-LINEAR DLS SCALING, INNINGS-1 TERMINAL BOUNDARY,
#           AND WICKETS-AWARE HIERARCHICAL OUTCOME DISTRIBUTIONS ──
def fit_par_and_term_inn1(train_df):
    inn1_m = (train_df["innings"] == 1) & (train_df["inn_total_balls"] == 120) & (train_df["legal_before"].between(0, 120))
    pt = train_df[inn1_m].groupby(["legal_before", "wkts_before"])["score_before"].mean().unstack()
    pt = pt.reindex(index=range(0, 121), columns=range(0, 11)).interpolate(method="linear", axis=0).ffill().bfill()
    for w in range(1, 11):
        pt[w] = np.minimum(pt[w], pt[w-1] - 2.5)
    par_grid = pt.values

    # Fit terminal Innings-1 Win Probability model P(Win | Final_RPO, is_t20i) on completed Innings-1 totals (Priority 6 fix)
    m_inn1 = train_df[train_df["innings"] == 1].drop_duplicates("match_id")
    rpo_f = m_inn1["first_total"].values * 6.0 / np.maximum(m_inn1["inn_total_balls"].values, 30.0)
    X_t = np.column_stack([np.ones(len(m_inn1)), rpo_f, m_inn1["is_t20i"].values])
    y_t = m_inn1["bat_team_won"].values
    def obj_t(w):
        p = expit(X_t @ w)
        return -np.sum(y_t * np.log(np.clip(p, 1e-9, 1-1e-9)) + (1 - y_t) * np.log(np.clip(1 - p, 1e-9, 1-1e-9))) + 0.5 * np.sum(w[1:]**2)
    w_term = minimize(obj_t, np.zeros(3), method="BFGS").x
    return par_grid, w_term

# Non-linear DLS resource-adjusted par lookup (Problem 5 fix: accounts for wickets remaining & non-linear overs exponent!)
def lookup_par_nonlinear_dls(balls_left, wkts, inn_total_balls, par_grid):
    tot_b = np.maximum(inn_total_balls, 30.0)
    elapsed_frac = np.clip((tot_b - balls_left) / tot_b, 0.0, 1.0)
    b_idx = np.clip(np.round(elapsed_frac * 120.0).astype(int), 0, 120)
    w_idx = np.clip(np.round(wkts).astype(int), 0, 10)
    # Non-linear DLS resource factor (tot_b / 120)^0.85 * wickets decay
    dls_scale = (tot_b / 120.0) ** 0.85
    return par_grid[b_idx, w_idx] * dls_scale

class PureStateWP:
    def __init__(self, inn=1, l2=10.0):
        self.inn = inn
        self.l2 = l2

    def _basis(self, X):
        if self.inn == 1:
            sc = X[:, 0] / 160.0
            wk = X[:, 1] / 10.0
            bl = X[:, 2] / np.maximum(X[:, 3], 30.0)
            t20i = X[:, 4]
            ko = X[:, 5]
            smp = X[:, 6] / 30.0
            return np.column_stack([
                np.ones(len(X)),
                sc, wk, bl, t20i, ko, smp,
                smp * (1.0 - bl),
                wk * (1.0 - bl),
                wk ** 2,
                sc / (1.05 - bl),
                (wk >= 1.0).astype(float)
            ])
        else:
            rn = X[:, 0]
            wk = X[:, 1]
            bl = np.maximum(X[:, 2], 1.0)
            tot_b = np.maximum(X[:, 3], 30.0)
            t20i = X[:, 4]
            ko = X[:, 5]
            rrr = np.clip((rn * 6.0) / bl, -5.0, 36.0)
            wkts_left = np.clip(10.0 - wk, 0.0, 10.0)
            res = (wkts_left / 10.0) ** 0.65 * (bl / tot_b) ** 0.55
            return np.column_stack([
                np.ones(len(X)),
                rn / 100.0,
                wk / 10.0,
                (wk / 10.0) ** 2,
                bl / tot_b,
                rrr / 10.0,
                (rrr / 10.0) ** 2,
                np.maximum(rrr - 10.0, 0.0) / 10.0,
                np.maximum(rrr - 14.0, 0.0) / 10.0,
                (rn / 160.0) - res,
                rrr * (wk / 10.0),
                (rn <= 6.0).astype(float) * (wkts_left >= 2.0).astype(float),
                t20i, ko
            ])

    def fit(self, X, y):
        B = self._basis(X)
        m = B.shape[1]
        def obj(w):
            z = B @ w
            p = expit(z)
            nll = -np.sum(y * np.log(np.clip(p, 1e-9, 1-1e-9)) + (1 - y) * np.log(np.clip(1 - p, 1e-9, 1-1e-9)))
            reg = 0.5 * self.l2 * np.sum(w[1:] ** 2)
            grad = B.T @ (p - y)
            grad[1:] += self.l2 * w[1:]
            return nll + reg, grad
        res = minimize(obj, np.zeros(m), jac=True, method="L-BFGS-B", options={"maxiter": 150})
        self.w_ = res.x
        return self

    def predict_proba(self, X):
        B = self._basis(X)
        p1 = expit(B @ self.w_)
        return np.column_stack([1.0 - p1, p1])

def shift_state(X, i, r, w, bl_drop, par_grid=None):
    N = X.copy()
    N[:, 1] = np.clip(N[:, 1] + w, 0.0, 10.0)
    N[:, 2] = np.where(N[:, 1] >= 10.0, 0.0, np.clip(N[:, 2] - bl_drop, 0.0, None))
    if i == 1:
        N[:, 0] += r
        N[:, 6] = N[:, 0] - lookup_par_nonlinear_dls(N[:, 2], N[:, 1], N[:, 3], par_grid)
    else:
        N[:, 0] -= r
    return N

def terminal(wp, N, i, w_term=None):
    if i == 1 and w_term is not None:
        is_done = (N[:, 1] >= 10.0) | (N[:, 2] <= 0.0)
        final_rpo = N[:, 0] * 6.0 / np.maximum(N[:, 3], 30.0)
        wp_term = expit(w_term[0] + w_term[1] * final_rpo + w_term[2] * N[:, 4])
        wp = np.where(is_done, wp_term, wp)
    elif i == 2:
        wp = np.where(N[:, 0] <= 0, 1.0, wp)
        wp = np.where((N[:, 1] >= 10) | ((N[:, 2] <= 0) & (N[:, 0] > 0)), 0.0, wp)
    return wp

# Problem 1 Fix: Hierarchical Dirichlet-smoothed outcome distributions conditioned on Format, Phase, WicketsLeft, and RequiredRuns
def fit_smoothed_outcome_dist(lg_df, fine_col, coarse_col, prior_m=300.0):
    coarse_probs = lg_df.groupby(coarse_col)["o"].value_counts(normalize=True).unstack().fillna(0.0)
    fine_counts  = lg_df.groupby([fine_col, coarse_col])["o"].value_counts().unstack().fillna(0.0)
    smoothed = {}
    for (f_key, c_key), row in fine_counts.iterrows():
        n_tot = row.sum()
        c_p = coarse_probs.loc[c_key] if c_key in coarse_probs.index else coarse_probs.mean()
        smoothed[f_key] = (row + prior_m * c_p) / (n_tot + prior_m)
    return pd.DataFrame(smoothed).T.fillna(0.0), coarse_probs

d0 = df[(df["balls_left_before"] > 0) & df["bat_team_won"].notna()].copy().reset_index(drop=True)
d0["wkts_bin"] = pd.cut(d0["wkts_before"], bins=[-1, 2, 5, 10], labels=["w0_2", "w3_5", "w6_9"]).astype(str)
d0["rn_bin"] = pd.cut(d0["runs_needed_before"].fillna(0), bins=[-100, 10, 20, 40, 1000], labels=["<10", "10-20", "20-40", ">40"]).astype(str)

d0["pk_coarse_1"] = d0["is_t20i"].astype(str) + "_" + d0["phase"].astype(str)
d0["pk_fine_1"]   = d0["pk_coarse_1"] + "_" + d0["wkts_bin"]

d0["pk_coarse_2"] = d0["is_t20i"].astype(str) + "_" + d0["phase"].astype(str) + "_" + d0["rn_bin"]
d0["pk_fine_2"]   = d0["pk_coarse_2"] + "_" + d0["wkts_bin"]

OUTS_ALL = [("W", 0, 1, 1), ("E", 1, 0, 0)] + [(str(r), r, 0, 1) for r in [0, 1, 2, 3, 4, 6]]

wp_parts = []
for i in (1, 2):
    d = d0[d0["innings"] == i].copy().reset_index(drop=True)
    y = d["bat_team_won"].astype(int).values
    wp_b, wp_a_actual, wp_a_nowkt, wp_a_bat_nb, lev, edw_all, edw_legal = (np.zeros(len(d)) for _ in range(7))
    fine_col = "pk_fine_1" if i == 1 else "pk_fine_2"
    coarse_col = "pk_coarse_1" if i == 1 else "pk_coarse_2"

    for tr, te in group_kfold_split(d["match_id"].values, 5):
        T_tr = d.iloc[tr]; T_te = d.iloc[te]
        par_grid_tr, w_term_tr = fit_par_and_term_inn1(T_tr) if i == 1 else (None, None)

        lg_tr = T_tr[~T_tr["is_bye_legbye"]].copy()
        lg_tr["o"] = np.where(
            ~lg_tr["legal_ball"].astype(bool), "E",
            np.where(lg_tr["is_wicket"].astype(bool), "W", lg_tr["runs_bat"].clip(upper=6).replace({5: 4}).astype(str))
        )
        p_out_all, p_coarse_all = fit_smoothed_outcome_dist(lg_tr, fine_col, coarse_col, prior_m=300.0)
        p_out_leg, p_coarse_leg = fit_smoothed_outcome_dist(lg_tr[lg_tr["legal_ball"]], fine_col, coarse_col, prior_m=300.0)

        if i == 1:
            smp_tr = T_tr["score_before"].values - lookup_par_nonlinear_dls(T_tr["balls_left_before"].values, T_tr["wkts_before"].values, T_tr["inn_total_balls"].values, par_grid_tr)
            smp_te = T_te["score_before"].values - lookup_par_nonlinear_dls(T_te["balls_left_before"].values, T_te["wkts_before"].values, T_te["inn_total_balls"].values, par_grid_tr)
            X_tr = np.column_stack([T_tr["score_before"], T_tr["wkts_before"], T_tr["balls_left_before"], T_tr["inn_total_balls"], T_tr["is_t20i"], T_tr["knock_out"], smp_tr]).astype(float)
            X_te = np.column_stack([T_te["score_before"], T_te["wkts_before"], T_te["balls_left_before"], T_te["inn_total_balls"], T_te["is_t20i"], T_te["knock_out"], smp_te]).astype(float)
        else:
            X_tr = np.column_stack([T_tr["runs_needed_before"], T_tr["wkts_before"], T_tr["balls_left_before"], T_tr["inn_total_balls"], T_tr["is_t20i"], T_tr["knock_out"]]).astype(float)
            X_te = np.column_stack([T_te["runs_needed_before"], T_te["wkts_before"], T_te["balls_left_before"], T_te["inn_total_balls"], T_te["is_t20i"], T_te["knock_out"]]).astype(float)

        mdl = PureStateWP(inn=i, l2=10.0).fit(X_tr, y[tr])
        p0 = mdl.predict_proba(X_te)[:, 1]

        Na = shift_state(X_te, i, T_te["approx_runs"].values, T_te["is_wicket"].astype(int).values, T_te["legal_ball"].astype(int).values, par_grid_tr)
        wp_a_actual[te] = terminal(mdl.predict_proba(Na)[:, 1], Na, i, w_term_tr)

        Na_nw = shift_state(X_te, i, T_te["approx_runs"].values, np.zeros(len(te)), T_te["legal_ball"].astype(int).values, par_grid_tr)
        wp_a_nowkt[te] = terminal(mdl.predict_proba(Na_nw)[:, 1], Na_nw, i, w_term_tr)

        # Problem 4 Fix: A No-Ball does NOT consume a legal delivery -> bl_drop = 0 (np.zeros(len(te)))!
        Na_nb = shift_state(X_te, i, T_te["runs_bat"].values, T_te["striker_out"].astype(int).values, np.zeros(len(te)), par_grid_tr)
        wp_a_bat_nb[te] = terminal(mdl.predict_proba(Na_nb)[:, 1], Na_nb, i, w_term_tr)

        L = np.zeros(len(te)); E_all = np.zeros(len(te)); E_leg = np.zeros(len(te))
        for name, r, w, bl_d in OUTS_ALL:
            Nh = shift_state(X_te, i, r, w, bl_d, par_grid_tr)
            p1 = terminal(mdl.predict_proba(Nh)[:, 1], Nh, i, w_term_tr)
            if name in p_out_all.columns:
                fb_a = T_te[coarse_col].map(p_coarse_all[name]).fillna(p_out_all[name].mean())
                pr_all = T_te[fine_col].map(p_out_all[name]).fillna(fb_a).values
            else:
                pr_all = np.zeros(len(te))
            L += pr_all * np.abs(p1 - p0)
            E_all += pr_all * (p1 - p0)
            if name != "E" and name in p_out_leg.columns:
                fb_l = T_te[coarse_col].map(p_coarse_leg[name]).fillna(p_out_leg[name].mean())
                pr_leg = T_te[fine_col].map(p_out_leg[name]).fillna(fb_l).values
                E_leg += pr_leg * (p1 - p0)

        wp_b[te], lev[te], edw_all[te], edw_legal[te] = p0, L, E_all, E_leg

    d["wp_before"] = wp_b
    d["wp_after"] = wp_a_actual
    d["wp_after_nowkt"] = wp_a_nowkt
    d["wp_after_bat_nb"] = wp_a_bat_nb
    d["leverage"] = lev
    d["exp_dwp_all"] = edw_all
    d["exp_dwp_legal"] = edw_legal

    print(f"Innings {i}: AUC {calc_auc(y, wp_b):.3f} | logloss {calc_logloss(y, wp_b):.3f} | Brier {calc_brier(y, wp_b):.4f}")
    bins = pd.qcut(wp_b, 5, duplicates="drop")
    cal = pd.DataFrame({"pred": wp_b, "actual": y, "bin": bins}).groupby("bin", observed=False)[["pred", "actual"]].mean().round(3)
    print(f"Innings {i} 5-Bin Calibration (Pred vs Actual):\n{cal.to_string()}\n")
    wp_parts.append(d)

wp = pd.concat(wp_parts, ignore_index=True)
del wp_parts; gc.collect()

wp["is_competitive"] = wp["wp_before"].between(0.03, 0.97)
wp["is_pressure"] = False
for inn in (1, 2):
    mask = wp["innings"] == inn
    q75 = wp.loc[mask & wp["is_competitive"], "leverage"].quantile(0.75)
    wp.loc[mask, "is_pressure"] = wp.loc[mask, "is_competitive"] & (wp.loc[mask, "leverage"] >= q75)

print(f"Pressure balls: {wp['is_pressure'].sum():,} | Normal balls: {(~wp['is_pressure']).sum():,}")
# Separate Super Over Summary (Problem 5 fix: report 6 Super Over matches separately rather than pooling with 120-ball par grid)
print(f"Super Over deliveries audited separately: {len(super_overs)} balls across {super_overs['match_id'].nunique()} matches")
wp.to_pickle(OUT / "wp_v4_master.pkl")


Innings 1: AUC 0.695 | logloss 0.632 | Brier 0.2212
Innings 1 5-Bin Calibration (Pred vs Actual):
                  pred  actual
bin                           
(0.0251, 0.359]  0.261   0.247
(0.359, 0.461]   0.414   0.422
(0.461, 0.544]   0.504   0.498
(0.544, 0.642]   0.590   0.622
(0.642, 0.984]   0.743   0.724

Innings 2: AUC 0.893 | logloss 0.410 | Brier 0.1330
Innings 2 5-Bin Calibration (Pred vs Actual):
                         pred  actual
bin                                  
(-0.000999305, 0.0902]  0.030   0.031
(0.0902, 0.327]         0.199   0.205
(0.327, 0.616]          0.469   0.458
(0.616, 0.861]          0.747   0.748
(0.861, 1.0]            0.939   0.944

Pressure balls: 45,560 | Normal balls: 153,284
Super Over deliveries audited separately: 38 balls across 6 matches


In [2]:
import gc, warnings, json
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import spearmanr

warnings.filterwarnings("ignore")

DATA = Path(r"C:\Users\Anmol Sriwastava\Desktop\RR_Analytics\data") if Path(r"C:\Users\Anmol Sriwastava\Desktop\RR_Analytics\data").exists() else (Path("/working_dir/c_e74e0d7d2df6d8f4/user_uploaded_files/ttl=63d/user_uploaded") if Path("/working_dir/c_e74e0d7d2df6d8f4/user_uploaded_files/ttl=63d/user_uploaded").exists() else Path("./data"))
OUT  = Path(r"C:\Users\Anmol Sriwastava\Desktop\RR_Analytics\Analysis_v4_final") if Path(r"C:\Users\Anmol Sriwastava\Desktop\RR_Analytics").exists() else Path("./Analysis_v4_final")
OUT.mkdir(exist_ok=True, parents=True)

wp = pd.read_pickle(OUT / "wp_v4_master.pkl")

FULL = {"India","Australia","England","South Africa","New Zealand","Pakistan","Sri Lanka","Bangladesh","West Indies","Afghanistan","Zimbabwe","Ireland"}
_mm = pd.read_csv(DATA/"matches.csv", usecols=["match_id","comp_id","match_date"]).merge(pd.read_csv(DATA/"competitions.csv", usecols=["comp_id","comp_name"]), on="comp_id")
_mm["year"] = pd.to_datetime(_mm["match_date"]).dt.year
_tn = pd.read_csv(DATA/"teams.csv").drop_duplicates("team_id").set_index("team_id")["team_name"]
_t = wp.drop_duplicates("match_id")[["match_id","batting_team_id","bowling_team_id"]].merge(_mm, on="match_id")
_t["ipl"] = _t.comp_name.str.startswith("IPL"); _t["minor"] = _t.comp_name.str.contains("Emerging|Rising")
_t["both_full"] = _t.batting_team_id.map(_tn).isin(FULL) & _t.bowling_team_id.map(_tn).isin(FULL)
_keep_main = set(_t.loc[_t.ipl | (~_t.minor & _t.both_full), "match_id"])
_keep_all_senior = set(_t.loc[~_t.minor, "match_id"])

def make_frames(df_in):
    dwp_actual = (df_in["wp_after"] - df_in["wp_before"]) * 100.0
    dwp_nowkt  = (df_in["wp_after_nowkt"] - df_in["wp_before"]) * 100.0
    dwp_bat_nb = (df_in["wp_after_bat_nb"] - df_in["wp_before"]) * 100.0
    exp_all    = df_in["exp_dwp_all"] * 100.0
    exp_legal  = df_in["exp_dwp_legal"] * 100.0
    WINSOR_CAP = 8.0

    b0 = df_in[(~df_in["is_wide"]) & (~df_in["is_bye_legbye"])].copy()
    bat_raw_x = np.where(
        b0["is_noball"].values,
        dwp_bat_nb.loc[b0.index].values - exp_legal.loc[b0.index].values,
        np.where(
            b0["is_wicket"].astype(bool).values & (~b0["striker_out"].astype(bool).values),
            dwp_nowkt.loc[b0.index].values - exp_legal.loc[b0.index].values,
            dwp_actual.loc[b0.index].values - exp_legal.loc[b0.index].values
        )
    )
    b0["x"] = np.clip(np.nan_to_num(bat_raw_x, nan=0.0), -WINSOR_CAP, WINSOR_CAP)
    b0["pid"] = b0["striker_id"]
    b0["is_p"] = b0["is_pressure"]

    w0 = df_in[~df_in["is_bye_legbye"]].copy()
    bow_raw_x = np.where(
        w0["is_wicket"].astype(bool).values & (~w0["bowler_wkt"].astype(bool).values),
        -(dwp_nowkt.loc[w0.index].values - exp_all.loc[w0.index].values),
        -(dwp_actual.loc[w0.index].values - exp_all.loc[w0.index].values)
    )
    w0["x"] = np.clip(np.nan_to_num(bow_raw_x, nan=0.0), -WINSOR_CAP, WINSOR_CAP)
    w0["pid"] = w0["bowler_id"]
    w0["is_p"] = w0["is_pressure"]
    return b0, w0

wv = wp[wp.match_id.isin(_keep_main)].copy()
for inn in (1, 2):
    mask = wv["innings"] == inn
    q75 = wv.loc[mask & wv["is_competitive"], "leverage"].quantile(0.75)
    wv.loc[mask, "is_pressure"] = wv.loc[mask, "is_competitive"] & (wv.loc[mask, "leverage"] >= q75)
bat0, bow0 = make_frames(wv)

# ── STEP 4 & 5: BIVARIATE EMPIRICAL BAYES RESIDUAL R_{i,EB} + STANDALONE CONSISTENCY (C_i) & DOWNSIDE (D_i) ──
def prep_match_level(frame):
    is_p = frame["is_p"].values.astype(bool)
    p = frame[is_p]; n = frame[~is_p]
    var_p_ball = float(p["x"].var(ddof=1))
    var_n_ball = float(n["x"].var(ddof=1))

    pm = p.groupby(["match_id", "pid"])["x"].agg(n="size", sx="sum").reset_index()
    pm["m_mean"] = pm["sx"] / pm["n"]
    pm["pos_m"] = (pm["sx"] > 0).astype(float)
    nm = n.groupby(["match_id", "pid"])["x"].agg(nn="size", snx="sum").reset_index()
    return pm, nm, var_p_ball, var_n_ball

def build_eb_v5(pm, nm, var_p_ball, var_n_ball, min_p=100, min_n=100, wt_dict=None, beta_override=None, kappa=10.0):
    if wt_dict is not None:
        wp_m = pm["match_id"].map(wt_dict).fillna(0.0).values
        wn_m = nm["match_id"].map(wt_dict).fillna(0.0).values
        eff_w = wp_m * (pm["n"].values / (pm["n"].values + 4.0))
        pm_w = pd.DataFrame({
            "pid": pm["pid"].values,
            "n": pm["n"].values * wp_m,
            "sx": pm["sx"].values * wp_m,
            "matches_p": wp_m,
            "eff_matches": eff_w,
            "pos_matches": pm["pos_m"].values * eff_w,
            "sum_down2": (np.minimum(pm["m_mean"].values, 0.0) ** 2) * eff_w
        })
        nm_w = pd.DataFrame({
            "pid": nm["pid"].values,
            "nn": nm["nn"].values * wn_m,
            "snx": nm["snx"].values * wn_m
        })
    else:
        eff_w = pm["n"].values / (pm["n"].values + 4.0)
        pm_w = pd.DataFrame({
            "pid": pm["pid"].values,
            "n": pm["n"].values.astype(float),
            "sx": pm["sx"].values,
            "matches_p": np.ones(len(pm)),
            "eff_matches": eff_w,
            "pos_matches": pm["pos_m"].values * eff_w,
            "sum_down2": (np.minimum(pm["m_mean"].values, 0.0) ** 2) * eff_w
        })
        nm_w = pd.DataFrame({
            "pid": nm["pid"].values,
            "nn": nm["nn"].values.astype(float),
            "snx": nm["snx"].values
        })

    T = pm_w.groupby("pid").sum().join(nm_w.groupby("pid").sum(), how="inner").fillna(0.0)
    T = T[(T["n"] > 0) & (T["nn"] > 0)].copy()
    T["P"] = T["sx"] / T["n"]
    T["N"] = T["snx"] / T["nn"]

    b_bar_p = (T["n"] / np.maximum(T["matches_p"], 1.0)).clip(1.0, 15.0)
    deff_p = 1.0 + 0.03 * (b_bar_p - 1.0)
    v_p = (var_p_ball * deff_p) / T["n"]
    v_n = var_n_ball / T["nn"]

    elig = T[(T["n"] >= min_p) & (T["nn"] >= min_n)]
    if len(elig) < 5:
        elig = T[(T["n"] >= 30) & (T["nn"] >= 30)]

    # 1. Empirical Bayes shrinkage of Normal baseline N -> N_eb
    tau2_n = max(float(elig["N"].var(ddof=1) - v_n.loc[elig.index].median()), 0.005)
    B_n = tau2_n / (tau2_n + v_n)
    mu_n = float(elig["N"].mean())
    T["N_eb"] = mu_n + B_n * (T["N"] - mu_n)

    # 2. WLS Regression with Explicit Intercept alpha_hat and Disattenuated Slope beta_hat
    w_reg = 1.0 / np.maximum(v_p.loc[elig.index].values + elig["P"].var(), 1e-6)
    w_reg /= w_reg.sum()
    x_e, y_e = elig["N"].values, elig["P"].values
    mx = float(np.sum(w_reg * x_e))
    my = float(np.sum(w_reg * y_e))
    vx = float(np.sum(w_reg * (x_e - mx) ** 2))
    if beta_override is not None:
        beta_hat = float(beta_override)
    else:
        beta_hat = float(np.sum(w_reg * (x_e - mx) * (y_e - my)) / max(vx, 1e-8))
    alpha_hat = float(my - beta_hat * mx)

    T["alpha_hat"] = alpha_hat
    T["beta_hat"] = beta_hat

    # Baseline expectation P_pred = alpha_hat + beta_hat * N_eb
    # Residual R_i = P_i - (alpha_hat + beta_hat * max(N_eb, 0)), shrunk toward prior_r = beta_hat * min(N_eb, 0)
    N_ref = np.maximum(T["N_eb"], 0.0)
    T["P_pred"] = alpha_hat + beta_hat * N_ref
    T["R"] = T["P"] - T["P_pred"]
    T["V_R"] = v_p + (beta_hat ** 2) * (B_n ** 2) * v_n

    elig_r = T.loc[elig.index, "R"]
    elig_vr = T.loc[elig.index, "V_R"]
    tau2_r = max(float(elig_r.var(ddof=1) - elig_vr.median()), 0.025)

    B_r = tau2_r / (tau2_r + T["V_R"])
    prior_r = beta_hat * np.minimum(T["N_eb"], 0.0)
    T["R_eb"] = prior_r + B_r * (T["R"] - prior_r)
    T["psd_R"] = np.sqrt(B_r * T["V_R"])

    # 3. Standalone Reliability Metrics (Reported Separately AND Tested in Composite):
    # C_i (%): Exposure-weighted Positive-Match Frequency (Beta-Binomial EB)
    p0_rate = float(elig["pos_matches"].sum() / np.maximum(elig["eff_matches"].sum(), 1.0))
    T["C_i_raw"] = (T["pos_matches"] / np.maximum(T["eff_matches"], 1.0)) * 100.0
    T["C_i_eb"]  = ((T["pos_matches"] + kappa * p0_rate) / (T["eff_matches"] + kappa)) * 100.0

    # D_i: Downside Semi-Deviation across matches (shrunk toward role median)
    d_raw = np.sqrt(np.maximum(T["sum_down2"] / np.maximum(T["eff_matches"], 1.0), 0.0))
    d_prior = float(np.median(d_raw.loc[elig.index]))
    T["D_i_eb"] = (T["eff_matches"] * d_raw + 12.0 * d_prior) / (T["eff_matches"] + 12.0)

    # Composite Reliability Score (for comparison alongside pure R_eb ranking)
    cons_lift = 0.006 * (T["C_i_eb"] - p0_rate * 100.0)
    vol_pen   = 0.050 * np.maximum(T["D_i_eb"] - d_prior, 0.0)
    T["reliability_score"] = T["R_eb"] + cons_lift - vol_pen
    return T

def evaluate_role(frame, namecol, min_p, min_n, rank_col="R_eb"):
    pm, nm, vp, vn = prep_match_level(frame)
    T = build_eb_v5(pm, nm, vp, vn, min_p=min_p, min_n=min_n)
    T = T[(T.n >= min_p) & (T.nn >= min_n)].copy()
    T["name"] = frame.groupby("pid")[namecol].agg(lambda s: s.mode()[0]).reindex(T.index)
    T = T.sort_values(rank_col, ascending=False).reset_index()

    ms = frame["match_id"].unique()
    rng = np.random.default_rng(7)
    top_hits = {}; boot_vals = {}
    for b in range(300):
        sampled = rng.choice(ms, len(ms), replace=True)
        wt = pd.Series(sampled).value_counts().to_dict()
        Tb = build_eb_v5(pm, nm, vp, vn, min_p=min_p, min_n=min_n, wt_dict=wt)
        Tb = Tb[(Tb.n >= min_p) & (Tb.nn >= min_n)]
        for pid in Tb.sort_values(rank_col, ascending=False).head(10).index:
            top_hits[pid] = top_hits.get(pid, 0) + 1
        for pid, val in Tb[rank_col].items():
            boot_vals.setdefault(pid, []).append(val)

    T["boot_top10_pct"] = T["pid"].map(pd.Series(top_hits) / 300.0 * 100.0).fillna(0.0).round(0)
    T["ci90_lo"] = T["pid"].map({k: np.percentile(v, 5) for k, v in boot_vals.items()}).round(3)
    T["ci90_hi"] = T["pid"].map({k: np.percentile(v, 95) for k, v in boot_vals.items()}).round(3)
    return T

MIN_P_BAT, MIN_N_BAT = 100, 100
MIN_P_BOWL, MIN_N_BOWL = 120, 120

bt_reb = evaluate_role(bat0, "striker_name", MIN_P_BAT, MIN_N_BAT, rank_col="R_eb")
bo_reb = evaluate_role(bow0, "bowler_name", MIN_P_BOWL, MIN_N_BOWL, rank_col="R_eb")

bt_reb.to_csv(OUT / "final_batters_v5.csv", index=False)
bo_reb.to_csv(OUT / "final_bowlers_v5.csv", index=False)

print("Estimated WLS Baseline Parameters (Unconstrained):")
print(f"  Batters: alpha_hat = {bt_reb['alpha_hat'].iloc[0]:+.3f}, beta_hat = {bt_reb['beta_hat'].iloc[0]:+.3f}")
print(f"  Bowlers: alpha_hat = {bo_reb['alpha_hat'].iloc[0]:+.3f}, beta_hat = {bo_reb['beta_hat'].iloc[0]:+.3f}\n")

cols = ["name","n","matches_p","P","N","R_eb","ci90_lo","ci90_hi","boot_top10_pct","C_i_eb","D_i_eb","reliability_score"]
print("=== FINAL TOP 10 BATTERS (Ranked by Pure Shrunk Residual R_eb, with Standalone C_i & D_i) ===")
print(bt_reb.head(10)[cols].round(3).to_string(index=False))

print("\n=== FINAL TOP 10 BOWLERS (Ranked by Pure Shrunk Residual R_eb, with Standalone C_i & D_i) ===")
print(bo_reb.head(10)[cols].round(3).to_string(index=False))

# ── VALIDATION & SENSITIVITY SUITE ──
base_bat = set(bt_reb.head(10)["name"])
base_bowl = set(bo_reb.head(10)["name"])

tb_comp = set(bt_reb.sort_values("reliability_score", ascending=False).head(10)["name"])
tw_comp = set(bo_reb.sort_values("reliability_score", ascending=False).head(10)["name"])
print(f"\nPure R_eb vs Composite Reliability Score Top-10 Overlap: Batters {len(tb_comp & base_bat)}/10 | Bowlers {len(tw_comp & base_bowl)}/10")

wv_all = wp[wp.match_id.isin(_keep_all_senior)].copy()
for inn in (1, 2):
    mask = wv_all["innings"] == inn
    q75 = wv_all.loc[mask & wv_all["is_competitive"], "leverage"].quantile(0.75)
    wv_all.loc[mask, "is_pressure"] = wv_all.loc[mask, "is_competitive"] & (wv_all.loc[mask, "leverage"] >= q75)
bat_all, bow_all = make_frames(wv_all)
pm_ba, nm_ba, vp_ba, vn_ba = prep_match_level(bat_all)
T_ba = build_eb_v5(pm_ba, nm_ba, vp_ba, vn_ba, MIN_P_BAT, MIN_N_BAT)
T_ba = T_ba[(T_ba.n >= MIN_P_BAT) & (T_ba.nn >= MIN_N_BAT)].copy()
T_ba["name"] = bat_all.groupby("pid")["striker_name"].agg(lambda s: s.mode()[0]).reindex(T_ba.index)
top_ba = set(T_ba.sort_values("R_eb", ascending=False).head(10)["name"])

pm_wa, nm_wa, vp_wa, vn_wa = prep_match_level(bow_all)
T_wa = build_eb_v5(pm_wa, nm_wa, vp_wa, vn_wa, MIN_P_BOWL, MIN_N_BOWL)
T_wa = T_wa[(T_wa.n >= MIN_P_BOWL) & (T_wa.nn >= MIN_N_BOWL)].copy()
T_wa["name"] = bow_all.groupby("pid")["bowler_name"].agg(lambda s: s.mode()[0]).reindex(T_wa.index)
top_wa = set(T_wa.sort_values("R_eb", ascending=False).head(10)["name"])

print(f"Competition Filter Sensitivity (Full-Member vs All T20I): Batters {len(top_ba & base_bat)}/10 | Bowlers {len(top_wa & base_bowl)}/10")
print(f"Share of pressure balls in competitive win-prob corridor [20%, 80%]: {wv.loc[wv['is_pressure'], 'wp_before'].between(0.20, 0.80).mean()*100:.1f}% (vs {wv.loc[~wv['is_pressure'], 'wp_before'].between(0.20, 0.80).mean()*100:.1f}% for normal balls)")

def eval_custom(frame, namecol, mp, mn, q=0.75, beta_override=None):
    f = frame.copy()
    lab_map = wv[wv["is_competitive"]].groupby("innings")["leverage"].quantile(q).to_dict()
    f["is_p"] = f["is_competitive"] & (f["leverage"].values >= f["innings"].map(lab_map).values)
    pm, nm, vp, vn = prep_match_level(f)
    T = build_eb_v5(pm, nm, vp, vn, mp, mn, beta_override=beta_override)
    T = T[(T.n >= mp) & (T.nn >= mn)].copy()
    T["name"] = f.groupby("pid")[namecol].agg(lambda s: s.mode()[0]).reindex(T.index)
    return T.sort_values("R_eb", ascending=False).reset_index()

print("\n--- TEST 1: PRESSURE THRESHOLD SWEEP (R_eb) ---")
for q in (0.65, 0.70, 0.75, 0.80, 0.85):
    tb = set(eval_custom(bat0, "striker_name", MIN_P_BAT, MIN_N_BAT, q=q).head(10)["name"])
    tw = set(eval_custom(bow0, "bowler_name", MIN_P_BOWL, MIN_N_BOWL, q=q).head(10)["name"])
    print(f"  Top {round((1-q)*100):2d}% pressure (q={q}): Batters {len(tb & base_bat):2d}/10 | Bowlers {len(tw & base_bowl):2d}/10")

print("\n--- TEST 2: MINIMUM BALLS SWEEP (R_eb) ---")
for mp in (80, 100, 120, 150, 200):
    tb = set(eval_custom(bat0, "striker_name", mp, mp, q=0.75).head(10)["name"])
    tw = set(eval_custom(bow0, "bowler_name", mp, mp, q=0.75).head(10)["name"])
    print(f"  Min {mp:3d} pressure & {mp:3d} normal balls: Batters {len(tb & base_bat):2d}/10 | Bowlers {len(tw & base_bowl):2d}/10")

print("\n--- TEST 3: BETA_HAT SENSITIVITY SWEEP (R_eb) ---")
for b_val in (0.0, 0.25, 0.50, 0.75):
    tb = set(eval_custom(bat0, "striker_name", MIN_P_BAT, MIN_N_BAT, beta_override=b_val).head(10)["name"])
    tw = set(eval_custom(bow0, "bowler_name", MIN_P_BOWL, MIN_N_BOWL, beta_override=b_val).head(10)["name"])
    print(f"  Beta override = {b_val:.2f}: Batters {len(tb & base_bat):2d}/10 | Bowlers {len(tw & base_bowl):2d}/10")

print("\n--- TEST 4: SPLIT-HALF RELIABILITY (Even vs Odd Match IDs) ---")
bat0["half"] = np.where(bat0["match_id"] % 2 == 0, "even", "odd")
bow0["half"] = np.where(bow0["match_id"] % 2 == 0, "even", "odd")
for role, f in [("Batters", bat0), ("Bowlers", bow0)]:
    for cut in (100, 150, 200):
        pm_e, nm_e, vpe, vne = prep_match_level(f[f["half"] == "even"])
        pm_o, nm_o, vpo, vno = prep_match_level(f[f["half"] == "odd"])
        Te = build_eb_v5(pm_e, nm_e, vpe, vne, cut, cut); Te = Te[(Te.n >= cut) & (Te.nn >= cut)]
        To = build_eb_v5(pm_o, nm_o, vpo, vno, cut, cut); To = To[(To.n >= cut) & (To.nn >= cut)]
        m = Te[["R_eb", "reliability_score"]].join(To[["R_eb", "reliability_score"]], how="inner", lsuffix="_e", rsuffix="_o")
        rho_r, _ = spearmanr(m["R_eb_e"], m["R_eb_o"])
        rho_rel, _ = spearmanr(m["reliability_score_e"], m["reliability_score_o"])
        ov = len(set(Te.sort_values("R_eb", ascending=False).head(10).index) & set(To.sort_values("R_eb", ascending=False).head(10).index))
        print(f"  {role:<7} Cutoff {cut} (n={len(m):<2}): R_eb Spearman {rho_r:+.3f} | Composite Spearman {rho_rel:+.3f} | Top-10 overlap: {ov}/10")

print("\n--- TEST 5: TEMPORAL SPLIT (2022-24 vs 2025-26, min 75 balls per era) ---")
bat0["year"] = bat0["match_id"].map(_mm.drop_duplicates("match_id").set_index("match_id")["year"])
bow0["year"] = bow0["match_id"].map(_mm.drop_duplicates("match_id").set_index("match_id")["year"])
for role, f in [("Batters", bat0), ("Bowlers", bow0)]:
    pm_e, nm_e, vpe, vne = prep_match_level(f[f["year"] < 2025])
    pm_l, nm_l, vpl, vnl = prep_match_level(f[f["year"] >= 2025])
    Te = build_eb_v5(pm_e, nm_e, vpe, vne, 75, 75); Te = Te[(Te.n >= 75) & (Te.nn >= 75)]
    Tl = build_eb_v5(pm_l, nm_l, vpl, vnl, 75, 75); Tl = Tl[(Tl.n >= 75) & (Tl.nn >= 75)]
    m = Te[["P", "R_eb", "reliability_score"]].join(Tl[["P", "R_eb", "reliability_score"]], how="inner", lsuffix="_e", rsuffix="_l")
    rho_r, _ = spearmanr(m["R_eb_e"], m["R_eb_l"])
    rho_rel, _ = spearmanr(m["reliability_score_e"], m["reliability_score_l"])
    print(f"  {role:<7} (n={len(m)} active in both eras): R_eb Spearman = {rho_r:+.3f} | Composite Spearman = {rho_rel:+.3f}")


Estimated WLS Baseline Parameters (Unconstrained):
  Batters: alpha_hat = +0.129, beta_hat = +0.608
  Bowlers: alpha_hat = -0.045, beta_hat = +0.575

=== FINAL TOP 10 BATTERS (Ranked by Pure Shrunk Residual R_eb, with Standalone C_i & D_i) ===
                name     n  matches_p     P      N  R_eb  ci90_lo  ci90_hi  boot_top10_pct  C_i_eb  D_i_eb  reliability_score
Vaibhav Sooryavanshi 168.0       23.0 1.163  0.447 0.275    0.220    0.797            83.0  69.551   1.163              0.355
    Heinrich Klaasen 204.0       35.0 0.873  0.184 0.238    0.103    0.692            72.0  59.815   0.805              0.259
         Harry Brook 149.0       23.0 0.989  0.096 0.237    0.071    0.765            62.0  62.666   1.388              0.271
      Mohammad Nawaz 123.0       20.0 0.885 -0.080 0.193    0.094    0.735            50.0  62.761   1.008              0.232
       Rovman Powell 268.0       42.0 0.613  0.054 0.192    0.064    0.507            49.0  62.799   1.288              0.231


# **Visuals**